# Cross-Layer Feature Tracking

Track how features evolve across layers: persistence, transformation,
creation/destruction events, lineage, and representation drift.

## Why This Matters

Feature cross layer tracking investigates the interpretable directions that models use internally. Understanding features — the natural units of neural computation — is essential for moving beyond neuron-level analysis to a true understanding of what models represent.

**Key references:**
- [Bricken et al. (2023) "Towards Monosemanticity"](https://transformer-circuits.pub/2023/monosemantic-features/index.html) — Sparse autoencoders find interpretable features
- [Cunningham et al. (2023) "Sparse Autoencoders Find Highly Interpretable Features"](https://arxiv.org/abs/2309.08600) — SAE features in larger language models

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.cross_layer_feature_tracking import (
    feature_persistence,
    transformation_analysis,
    creation_destruction_events,
    feature_lineage,
    representation_drift,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([0, 5, 10, 15, 20, 25, 30])
print("Model ready")

## Feature Persistence

In [ ]:
result = feature_persistence(model, tokens)
for l, (pers, mag) in enumerate(zip(result['persistence_scores'], result['projection_magnitudes'])):
    bar = '#' * int(float(pers) * 20)
    print(f"  Layer {l}: persistence={float(pers):.3f}, magnitude={float(mag):.4f} {bar}")
print(f"Peak layer: {result['peak_layer']}, decay_rate: {result['decay_rate']:.3f}")

## Transformation Analysis

In [ ]:
result = transformation_analysis(model, tokens)
for l in range(cfg.n_layers - 1):
    cos = float(result['inter_layer_cosines'][l])
    angle = float(result['rotation_angles'][l])
    stretch = float(result['stretch_factors'][l])
    print(f"  L{l}->L{l+1}: cos={cos:.3f}, angle={angle:.3f}rad, stretch={stretch:.3f}x")

## Creation/Destruction Events

In [ ]:
result = creation_destruction_events(model, tokens)
print(f"Creation layers: {result['creation_layers']}")
print(f"Destruction layers: {result['destruction_layers']}")
print(f"Net change: {result['net_change']}")
print(f"Dimension utilization: {[f'{float(u):.2f}' for u in result['dimension_utilization']]}")

## Feature Lineage

In [ ]:
result = feature_lineage(model, tokens, source_layer=0, n_directions=3)
for d_idx in range(3):
    pers = [f'{float(p):.3f}' for p in result['direction_persistence'][d_idx]]
    print(f"  Direction {d_idx}: {pers}")
print(f"Most persistent: dir {result['most_persistent_direction']}")

## Representation Drift

In [ ]:
tokens_list = [jnp.array([1, 2, 3, 4, 5, 6, 7]),
               jnp.array([50, 51, 52, 53, 54, 55, 56]),
               jnp.array([10, 20, 30, 40, 50, 60, 70])]
result = representation_drift(model, tokens_list)
for l, drift in enumerate(result['mean_drift_per_layer']):
    print(f"  L{l}->L{l+1}: drift={float(drift):.4f}")
print(f"Total drift: {result['total_drift']:.4f}")
print(f"Convergence: {result['convergence_rate']:.3f}")